In [25]:
import numpy as np
from scipy.sparse import eye
from scipy.sparse.linalg import expm, expm_multiply, norm
from utils import *

Consider QW on a 1D chain

In [69]:

n = 5
T = 1

# Ground truth
A = np.zeros((n,n))
for i in range(n-1):
    A[i,i+1] = 1
    A[i+1,i] = 1

U_A = expm(-1j * T * A)

# Hamiltonian embedding
codewords = get_codewords_1d(n, encoding="one-hot", periodic=False)
# bitstrings = get_bitstrings(n, 1, encoding="one-hot")
J_even = np.zeros((n,n))
J_odd = np.zeros((n,n))
for i in np.arange(0, n-1, 2):
    J_even[i,i+1] = 1
for i in np.arange(1, n-1, 2):
    J_odd[i,i+1] = 1
J = J_even + J_odd

print("-- One-hot w/o penalty --")
# Penalty free one-hot embedding
r = 5
A_even = np.zeros((n,n))
A_odd = np.zeros((n,n))
for i in np.arange(0, n-1, 2):
    A_even[i,i+1] = 1
    A_even[i+1,i] = 1
for i in np.arange(1, n-1, 2):
    A_odd[i,i+1] = 1
    A_odd[i+1,i] = 1

A_even = csc_matrix(A_even)
A_odd = csc_matrix(A_odd)
U_one_hot_subspace = eye(n)
for _ in range(r):
    U_one_hot_subspace = expm_multiply(-1j * (T/(2*r)) * A_even, U_one_hot_subspace)
    U_one_hot_subspace = expm_multiply(-1j * (T/r) * A_odd, U_one_hot_subspace)
    U_one_hot_subspace = expm_multiply(-1j * (T/(2*r)) * A_even, U_one_hot_subspace)

error = np.linalg.norm(U_A - U_one_hot_subspace, ord=2)
print("Penalty-free error:", error)

print("\n-- One-hot with penalty --")
# One-hot with penalty
g = 100
H_one_hot_pen = sum_J_xx(n, J) + g * (sum_h_z(n, np.ones(n)) - (n-2) * eye(2 ** n))
U_one_hot_pen = expm(-1j * T * H_one_hot_pen)
U_one_hot_pen_subspace = U_one_hot_pen[codewords][:,codewords]

# Leakage error
print("Leakage error:", 1 - np.abs(np.sum(U_one_hot_pen_subspace @ np.conj(U_one_hot_pen_subspace.T))) / n)
print("Fidelity:", np.abs(np.sum(U_one_hot_pen_subspace @ np.conj(U_A.T)) / n))
error = np.linalg.norm(U_A - U_one_hot_pen_subspace, ord=2)
print("Error:", error)

# Thrift
H_0 = sum_J_xx(n, J)
H_1 = (sum_h_z(n, np.ones(n)) - (n-2) * eye(2 ** n))
alpha = 1 / g
T_thrift = T * g
r = 5
U_one_hot_pen = eye(2 ** n)
for _ in range(r):
    U_one_hot_pen

U_one_hot_pen = expm_multiply(-1j * (T_thrift) * H_0, U_one_hot_pen)

-- One-hot w/o penalty --
Penalty-free error: 0.0062221509680694155

-- One-hot with penalty --
Leakage error: 4.592421761340226e-05
Fidelity: 0.9999761330272289
Error: 0.009506912382396789
